# Understanding the vorpal Pipeline

This notebook explains how vorpal's pipeline works internally — useful when you want to inspect a build that's already been run, understand why a stage re-ran (or didn't), edit chapter metadata, or debug a problem.

The central concept is the **manifest**: a single JSON file (`book.json`) that lives in the workdir and records every decision the pipeline made. Every stage reads from it and writes back to it. Understanding `book.json` is the key to understanding everything else.

We'll work with a real workdir if one is present; otherwise the notebook uses example data structures to illustrate each concept.

## The Manifest-Driven Design

Most tools that convert files between formats are linear pipelines: input goes in, output comes out, and if anything fails you start over. vorpal is different.

Every stage writes its results — and the decision record for how it produced them — into `book.json`. When a stage runs again, it first checks whether its inputs have changed. If nothing changed, it skips. If only one chapter changed (say, you edited its title in the manifest), only that chapter re-synthesizes.

The cache key for each synthesized audio chunk is derived from the chunk's text content, the voice name, and the playback speed. Changing the voice triggers a full re-synthesis; changing a chapter title only (not its text) does not re-synthesize the audio.

**What "resumable" means in practice:** if a synthesis run stops halfway through (power cut, OOM, Ctrl-C), you re-run `vorpal build` with identical arguments and it picks up from the last completed chunk. There is no "checkpoint" to restore — the completed chunks are simply already in the cache, so the pipeline skips them.

The manifest also serves as a communication channel between you and the pipeline. You can edit `book.json` directly — change a chapter title, flip an `include` flag, set a `spoken_intro` — and the next build picks up your changes.

In [ ]:
import json
import pathlib

# Try to find an existing workdir. Check common names.
candidate_dirs = sorted(pathlib.Path(".").glob("*_workdir"))
workdir = candidate_dirs[0] if candidate_dirs else None

manifest_data = None
if workdir:
    manifest_path = workdir / "book.json"
    if manifest_path.exists():
        manifest_data = json.loads(manifest_path.read_text(encoding="utf-8"))
        print(f"Loaded manifest from: {manifest_path}")

if manifest_data is None:
    # No real workdir present — create a minimal example to illustrate the structure
    print("No workdir found — showing example manifest structure.")
    manifest_data = {
        "schema_version": 2,
        "source": {
            "title": "The Time Machine",
            "author": "H. G. Wells",
            "file": "book.epub",
            "format": "epub",
            "content_hash": "sha256:abc123..."
        },
        "settings": {
            "voice": "af_heart",
            "speed": 1.0,
            "engine": "kokoro",
            "approved": False
        },
        "stages": {
            "extract": {"status": "done", "completed_at": "2026-06-11T10:00:00"},
            "segment": {"status": "done", "completed_at": "2026-06-11T10:00:05"},
            "normalize": {"status": "done", "completed_at": "2026-06-11T10:00:10"},
            "synthesize": {"status": "done", "completed_at": "2026-06-11T10:25:00"},
            "master": {"status": "done", "completed_at": "2026-06-11T10:26:30"}
        },
        "chapters": [
            {"id": "ch_001", "title": "Introduction", "kind": "front_matter", "include": False, "source": "outline", "confidence": 0.97, "words": 312},
            {"id": "ch_002", "title": "I. Introduction", "kind": "chapter", "include": True, "source": "outline", "confidence": 0.99, "words": 1840},
            {"id": "ch_003", "title": "II. The Machine", "kind": "chapter", "include": True, "source": "outline", "confidence": 0.99, "words": 2105}
        ]
    }

# Pretty-print source + settings + first 3 chapters
preview = {
    "schema_version": manifest_data.get("schema_version"),
    "source": manifest_data.get("source", {}),
    "settings": manifest_data.get("settings", {}),
    "stages": {k: v.get("status") for k, v in manifest_data.get("stages", {}).items()},
    "chapters_preview": manifest_data.get("chapters", [])[:3]
}
print(json.dumps(preview, indent=2))

## The 7 Stages

| Stage | Input | Output | Re-runs when |
|-------|-------|--------|------------------|
| **ingest** | Source file (EPUB/PDF/TXT) | `book.json` skeleton, raw file copy | Source file changes |
| **extract** | Raw file | `pages.jsonl` (text + geometry per page/block) | Source file hash changes |
| **segment** | `pages.jsonl` | Chapter boundaries written into `book.json` | Extract output changes, or `--stop-after segment` flag |
| **normalize** | Chapter text from manifest | Per-chapter `chunks.jsonl` files | Chapter text, voice, or speed changes |
| **synthesize** | `chunks.jsonl` per chapter | Per-chunk WAV files in `chunks/`, per-chapter WAV in `chapters/` | Chunk text+voice+speed cache key changes |
| **master** | Per-chapter WAVs | Final `.m4b` (and `.mp3`) file | Any chapter WAV changes, or metadata changes |
| **report** | Manifest + timing data | `report.md` summary | Always runs after master |

Stages are skipped by comparing content hashes, not timestamps. Touching a file does not re-run the stage; changing its content does.

## Stages 1–3: Getting the Text

The first three stages differ significantly by input format:

**EPUB** is the cleanest path. The EPUB container holds structured XHTML with a machine-readable table of contents (`toc.ncx` or `nav.xhtml`). Extract reads the spine items in order and produces clean text with block boundaries already marked. Chapter detection uses the NCX/nav TOC directly, giving confidence scores near 1.0 for most books.

**Born-digital PDF** (PDF with a text layer) goes through pdfminer/pdfplumber. Block geometry (x/y coordinates, font size, font weight) is preserved in `pages.jsonl` so the segmenter can identify headings by appearance rather than relying solely on text content. Two-page-spread layouts and fused header lines are handled by a merge pass.

**Scanned PDF** triggers the OCR path: each page is rasterized, preprocessed (deskew, binarize), and passed to Tesseract. Block geometry is recovered from Tesseract's hOCR output. OCR accuracy on clean 1960s+ book scans is typically 97–99%; earlier or lower-contrast scans may need review.

**TXT** is the simplest ingest: the file is read as UTF-8, split into lines, and blocks are inferred by blank-line boundaries. No geometry, no embedded TOC. Chapter detection relies entirely on heuristics (lines that look like headings by length and capitalization).

In [ ]:
# Show what pages.jsonl looks like — load first 2 entries from a real workdir,
# or show an example structure if none is present.

pages_path = workdir / "pages.jsonl" if workdir else None

if pages_path and pages_path.exists():
    print(f"Loading from: {pages_path}")
    entries = []
    with open(pages_path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 2:
                break
            entries.append(json.loads(line))
    print(json.dumps(entries, indent=2))
else:
    print("No pages.jsonl found — showing example entry structure:\n")
    example = [
        {
            "page": 1,
            "block": 0,
            "kind": "heading",
            "text": "I. Introduction",
            "font_size": 14.0,
            "bold": True,
            "x0": 72, "y0": 120, "x1": 310, "y1": 138
        },
        {
            "page": 1,
            "block": 1,
            "kind": "body",
            "text": "The Time Traveller (for so it will be convenient to speak of him) was expounding a recondite matter to us.",
            "font_size": 11.0,
            "bold": False,
            "x0": 72, "y0": 155, "x1": 524, "y1": 195
        }
    ]
    print(json.dumps(example, indent=2))

## Stage 4: Chapter Detection

The segmenter uses a three-level cascade. Each level produces chapter boundaries with a confidence score:

1. **Outline** — if the source has a machine-readable table of contents (EPUB NCX/nav, PDF outline/bookmarks), use it directly. Confidence 0.95–1.0. This is the most reliable path.

2. **Printed TOC** — scan the first N pages for a rendered table of contents ("Chapter One............12"-style lines with page numbers). Parse the titles and page numbers, then match them to the extracted text. Confidence 0.80–0.95.

3. **Heuristics** — use block geometry and text patterns to identify likely heading lines: short lines, larger/bolder font than body, isolated position, Roman numerals or "Chapter" keywords. Confidence 0.50–0.85.

All detected boundaries are written into `book.json` with their source, confidence score, word count, and an `include` flag. Boundaries below a configurable confidence threshold are flagged for review. The `vorpal review` command shows only the flagged and borderline chapters, so you're not reading through a hundred confident detections to find the two that need attention.

After the review gate, `vorpal review --approve` sets `settings.approved: true` in the manifest, telling the pipeline it's safe to proceed to synthesis.

In [ ]:
# Show the chapter table from the manifest
chapters = manifest_data.get("chapters", [])

if chapters:
    print(f"{'ID':<10} {'TITLE':<35} {'KIND':<15} {'INC':<5} {'SRC':<10} {'CONF':<6} {'WORDS'}")
    print("-" * 95)
    for c in chapters:
        inc = "yes" if c.get("include", True) else "no"
        conf = f"{c.get('confidence', 0):.2f}"
        title = c.get("title", "")[:34]
        print(f"{c.get('id',''):<10} {title:<35} {c.get('kind',''):<15} {inc:<5} {c.get('source',''):<10} {conf:<6} {c.get('words','')}")
else:
    print("No chapters in manifest.")

## Stage 5: Normalization and Chunking

Raw extracted text isn't ready for TTS. The normalize stage does several things:

**Text normalization:**
- Expands numbers: "1917" → "nineteen seventeen" (years) or "one thousand nine hundred seventeen" (counts), depending on context
- Expands abbreviations: "Dr." → "Doctor", "Vol." → "Volume"
- Handles em dashes, ellipses, and other punctuation that TTS engines treat inconsistently
- Strips page headers, running titles, and footnote markers that shouldn't be spoken

**Prosody-aware chunking** — splitting the text into pieces that synthesize well:

The chunker respects a strict hierarchy:
1. **Flush at paragraph boundaries** — never carry a chunk across a paragraph break. Paragraph breaks become natural pauses.
2. **Split at sentence boundaries only** — if a paragraph is too long for a single synthesis call, split at a sentence end. Never split a sentence across chunks.
3. **Never mid-sentence** — this rule is absolute. A mid-sentence split produces audible artifacts (pitch reset, unnatural pause). The chunker will produce an oversized chunk rather than break this rule.

Each chunk gets a content-addressed cache key: `sha256(text + voice + speed)`. The synthesis stage only calls the TTS engine for chunks whose key isn't already in the cache.

## Stage 6: TTS Synthesis

Synthesis is the longest stage. For each chapter, the synthesizer:

1. Iterates through the chapter's chunks
2. For each chunk, checks the WAV cache (`chunks/<hash>.wav`) — if present, uses it
3. For cache misses, calls the Kokoro TTS engine (GPU if available, CPU fallback)
4. Stitches the per-chunk WAVs into a per-chapter WAV using a short crossfade at chunk boundaries

**GPU vs CPU:** The Kokoro 82M model processes roughly 10–15 real-time-factor on a mid-range GPU (e.g., a full 19-hour audiobook in ~25 minutes). On CPU it's closer to 0.3–0.5x real-time — the same book takes 4–7 hours. If your machine has an NVIDIA GPU, make sure CUDA torch is installed (`python -c "import torch; print(torch.cuda.is_available())"`). vorpal will use it automatically.

**Crossfade stitching:** Each chunk boundary gets a 20ms crossfade — short enough to be inaudible but long enough to eliminate the click that appears when WAV files are concatenated without overlap. Paragraph boundaries get a slightly longer silence insert (200ms) to mark the paragraph break.

**The chunk cache is durable:** completed chunks survive a crash or a restart. If synthesis stops at chapter 7 chunk 43, restarting the build skips all 42 completed chunks and resumes from chunk 43. The cache is never invalidated unless the text, voice, or speed actually changes.

## Stage 7: Mastering

The mastering stage takes the per-chapter WAVs and produces the final M4B file.

**Loudness normalization (EBU R128):** Consumer audio varies wildly in loudness. Without normalization, a chapter synthesized from a long dense paragraph sounds louder than one from short crisp sentences — purely because of how TTS gain behaves with different text densities. The loudnorm pass runs two-pass EBU R128 normalization targeting -18 LUFS integrated loudness with a -1.0 dBTP true-peak ceiling. This is the broadcast standard used by Spotify, Apple Podcasts, and streaming audiobook platforms.

**M4B assembly via ffmpeg:**
1. Each normalized chapter WAV is encoded to AAC (128 kbps, 44.1 kHz mono)
2. All chapter AACs are concatenated using ffmpeg's `concat` demuxer
3. Chapter markers (title + start timestamp) are written as MP4 chapter atoms
4. Cover art (from the source file, or a default placeholder) is attached as a video stream
5. ID3-compatible metadata (title, author, narrator) is embedded

**MP3 side product:** A plain MP3 is also produced in the workdir for devices that don't support M4B. It doesn't have chapter markers but is otherwise identical audio.

**File sizes:** AAC at 128 kbps produces roughly 57 MB per hour of audio. The 19h16m Trotsky production came to 569 MB — consistent with this estimate.

## Inspecting a Build Report

After mastering, vorpal writes a `report.md` file to the workdir (and optionally to the current directory). The report covers:

- Source file, format, and content hash
- Voice, speed, engine settings
- Per-chapter table: title, words, duration, synthesis time, cache hit rate
- Total duration and file size
- Stage timing (how long each pipeline stage took)
- Any warnings (low-confidence chapter detections, OCR quality flags, skipped pages)

The report is a plain Markdown file — readable in any text editor, renderable in any Markdown viewer.

In [ ]:
import pathlib

# Look for a report: first in the current directory, then in the workdir
report_candidates = list(pathlib.Path(".").glob("*_report.md"))
if workdir:
    report_candidates += list(workdir.glob("report.md"))

# Also check for the Trotsky report that may be present in this repo
trotsky_report = pathlib.Path("trotsky_v1_report.md")
if trotsky_report.exists():
    report_candidates.insert(0, trotsky_report)

if report_candidates:
    report_path = report_candidates[0]
    print(f"Reading: {report_path}\n")
    print(report_path.read_text(encoding="utf-8")[:3000])
else:
    print("No report found — run a build first, then re-run this cell.")
    print()
    print("A report will appear at:")
    print("  <output_stem>_workdir/report.md")
    print("  (or <output_stem>_report.md in the current directory)")

## Editing the Manifest

`book.json` is designed to be edited by hand. The fields most commonly changed:

**`chapters[N].title`** — the chapter title as it will appear in the audiobook player's chapter list. Useful when OCR or heuristics produced an ugly heading (e.g., `"CHAPTER THE FIRST—IN WHICH THE HERO"` → `"Chapter One"`). Changing the title does not re-synthesize the audio.

**`chapters[N].include`** — set to `false` to exclude a chapter from synthesis. Use this for indexes, bibliographies, lists of illustrations, and other material that doesn't work as audio.

**`chapters[N].spoken_intro`** — a short string to synthesize and prepend before the chapter's own audio. Useful for adding chapter-number announcements: `"Chapter Three."` The spoken intro is synthesized with the same voice and settings as the rest of the book.

**`settings.voice`** — change this to switch the narrator for the next build.

**`settings.approved`** — set to `true` to approve the chapter structure and allow synthesis to proceed. Equivalent to running `vorpal review --approve`.

After editing, run:
```bash
vorpal review --approve  # marks the manifest approved
vorpal build book.epub   # re-runs from the first changed stage
```

Only the stages downstream of your change will re-run. If you only changed chapter titles (no text content), only the mastering stage re-runs (to update chapter marker names). If you changed `include` flags, synthesis of the changed chapters and mastering re-run. If you changed the voice, all synthesis re-runs.

## Fidelity Checking

For EPUB and TXT sources (where the input text is directly machine-readable), vorpal can verify that no body text was silently dropped during normalization or synthesis.

The fidelity checker (`vorpal/qa/fidelity.py`) compares the normalized text that was sent to the TTS engine against the original source text, using character-level similarity. A score of **1.000** means every character of body text made it through. Scores below ~0.995 indicate text was dropped or significantly transformed and should be investigated.

To run the fidelity check:
```bash
python -m vorpal.qa.fidelity my_first_book_workdir/
```

The output is a per-chapter similarity table, plus a global score. The check only runs on EPUB and TXT sources — scanned PDFs inherently involve OCR approximations, so character-level fidelity to the original image is not a meaningful metric.

The hard rule in vorpal's design: **body text is never silently dropped**. If the normalizer can't safely handle a passage, it flags it rather than skipping it. The fidelity check is the backstop that catches any case where this rule was violated.